In [ ]:
from pathlib import Path
import os

import numpy as np
import napari
from napari.utils.notebook_display import nbscreenshot

from infer_subc.core.file_io import (read_czi_image,
                                     export_inferred_organelle,
                                     list_image_files)
from infer_subc.core.img import *         

%load_ext autoreload
%autoreload 2

In [ ]:
##########################
#  infer_LYSOSOMES
##########################
def _infer_lyso(
                                in_img: np.ndarray,
                                lyso_ch: int,
                                median_sz: int,
                                gauss_sig: float,
                                dot_scale_1: float,
                                dot_cut_1: float,
                                dot_scale_2: float,
                                dot_cut_2: float,
                                dot_scale_3: float,
                                dot_cut_3: float,
                                dot_method: str,
                                fil_scale_1: float,
                                fil_cut_1: float,
                                fil_scale_2: float, 
                                fil_cut_2: float, 
                                fil_scale_3: float, 
                                fil_cut_3: float,
                                fil_method: str,
                                min_hole_w: int,
                                max_hole_w: int,
                                small_obj_w: int,
                                max_obj_size: int,
                                fill_filter_method: str
                            ) -> np.ndarray:
    """
    Procedure to infer lysosome from linearly unmixed input,
    
    Parameters
    ------------
    in_img: 
        a 3d image containing all the channels
    median_sz: 
        width of median filter for signal
    gauss_sig: 
        sigma for gaussian smoothing of  signal
    dot_scale: 
        scales (log_sigma) for dot filter (1,2, and 3)
    dot_cut: 
        threshold for dot filter thresholds (1,2,and 3)
    fil_scale: 
        scale (log_sigma) for filament filter
    fil_cut: 
        threshold for filament fitered threshold
    min_hole_w: 
        hole filling min for nuclei post-processing
    max_hole_w: 
        hole filling cutoff for nuclei post-processing
    small_obj_w: 
        minimu object size cutoff for nuclei post-processing
    fill_filter_method:
        to fill snall holes and remove small objects in "3D" or "slice-by-slice"

    Returns
    -------------
    lyso_object
        mask defined extent of lysosome object

    """
    ###################
    # EXTRACT
    ###################    
    lyso = select_channel_from_raw(in_img, lyso_ch)

     ###################
    # PRE_PROCESSING
    ###################    
    lyso1 =  scale_and_smooth(lyso,
                             median_size = median_sz, 
                             gauss_sigma = gauss_sig)
   ###################
    # CORE_PROCESSING
    ###################
    bw_dot = dot_filter_3(lyso1, dot_scale_1, dot_cut_1, dot_scale_2, dot_cut_2, dot_scale_3, dot_cut_3, dot_method)

    bw_filament = filament_filter_3(lyso1, fil_scale_1, fil_cut_1, fil_scale_2, fil_cut_2, fil_scale_3, fil_cut_3, fil_method)

    bw = np.logical_or(bw_dot, bw_filament)

    ###################
    # POST_PROCESSING
    ###################
    struct_obj = fill_and_filter_linear_size(bw, hole_min=min_hole_w, hole_max=max_hole_w, min_size=small_obj_w, method=fill_filter_method)
    def filter_max_size(in_img: np.ndarray,
                    max_size: int,
                    method: str = "3D",
                    connectivity: int = 1):
        sz_fltr = size_filter_linear_size(img=in_img, min_size=max_size, method=method, connectivity=connectivity)
        sz_fltr = sz_fltr > 0
        out_img = in_img * ~sz_fltr
        return out_img
    struct_obj = filter_max_size(struct_obj,
                               max_size=max_obj_size,
                               method=fill_filter_method)
    ###################
    # LABELING
    ###################
    struct_obj1 = label_uint16(struct_obj)

    return struct_obj1


In [ ]:
def batch_run(raw_path: str,
              out_path: str,
              raw_file_type: str,
              lyso_ch: int,
              median_sz: int,
              gauss_sig: float,
              dot_scale_1: float,
              dot_cut_1: float,
              dot_scale_2: float,
              dot_cut_2: float,
              dot_scale_3: float,
              dot_cut_3: float,
              dot_method: str,
              fil_scale_1: float,
              fil_cut_1: float,
              fil_scale_2: float, 
              fil_cut_2: float, 
              fil_scale_3: float, 
              fil_cut_3: float,
              fil_method: str,
              min_hole_w: int,
              max_hole_w: int,
              small_obj_w: int,
              max_obj_size: int,
              fill_filter_method: str
              ):
    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(out_path, str): out_path = Path(out_path)
    if not Path.exists(out_path):
        Path.mkdir(out_path)
        print(f"making {out_path}")
    img_file_list = list_image_files(raw_path, raw_file_type)
    out_files = []
    for img_f in img_file_list:
        print(f"Starting segmentation on {img_f}")
        img_data, meta_dict = read_czi_image(img_f)

        lyso_seg = _infer_lyso(img_data,
                                lyso_ch,
                                median_sz,
                                gauss_sig,
                                dot_scale_1,
                                dot_cut_1,
                                dot_scale_2,
                                dot_cut_2,
                                dot_scale_3,
                                dot_cut_3,
                                dot_method,
                                fil_scale_1,
                                fil_cut_1,
                                fil_scale_2, 
                                fil_cut_2, 
                                fil_scale_3, 
                                fil_cut_3,
                                fil_method,
                                min_hole_w,
                                max_hole_w,
                                small_obj_w,
                                max_obj_size,
                                fill_filter_method)
        
        out_file_n = export_inferred_organelle(lyso_seg, "lyso", meta_dict, out_path)
        out_files.append(out_file_n)
        print(f"Completed segmentation on {img_f}")

    return out_files

In [ ]:
fls = batch_run(raw_path = "",
                out_path = "",
                raw_file_type = "",
                lyso_ch = 0,
                median_sz = 0,
                gauss_sig = 1.34,
                dot_scale_1 = 0.00,
                dot_cut_1 = 0.00,
                dot_scale_2 = 0.00,
                dot_cut_2 = 0.00,
                dot_scale_3 = 0.00,
                dot_cut_3 = 0.00,
                dot_method = "",
                fil_scale_1 = 0.00,
                fil_cut_1 = 0.00,
                fil_scale_2 = 0.00, 
                fil_cut_2 = 0.00, 
                fil_scale_3 = 0.00, 
                fil_cut_3 = 0.00,
                fil_method = "",
                min_hole_w = 0,
                max_hole_w = 0,
                small_obj_w = 0,
                max_obj_size = 0,
                fill_filter_method = "")